# Programação funcional com Python

Este notebook tem por objetivo explorar os fundamentos de programação funcional e entender como Python lida com eles.

## Sumário
- [O que é Programação Funcional?](#o-que-é-programação-funcional)
- [Fundamentos de FP](#fundamentos-de-fp)
    - [Funções como *first-class citizens*](#funções-como-first-class-citizens)
    - [Funções puras e impuras](#funções-puras-e-impuras)
    - [Imutabilidade](#imutabilidade)
    - [Funções de ordem superior](#funções-de-ordem-superior)
    - [Composição de funções](#composição-de-funções)
    - [Aplicação parcial](#aplicação-parcial)
- [List Comprehensions e Generator Expressions](#list-comprehensions-e-generator-expressions)
- [Pipelines](#pipelines)
- [Limitações de FP em Python](#limitações-de-fp-em-python)
    - [Contexto Histórico](#contexto-histórico)
    - [Tail Call Optimization](#tail-call-optimization)
    - [Pattern matching](#pattern-matching)
    - [Mutabilidade por padrão](#mutabilidade-por-padrão)
    - [Lambdas limitadas](#lambdas-limitadas)
    - [Limitações de sintaxe](#limitações-de-sintaxe)
- [Consequências das limitações](#consequências-das-limitações)
- [Conclusão e Recomendações](#conclusão-e-recomendações)

## O que é Programação Funcional?
Programação funcional (FP) é um paradigma de programação em que programas são construídos aplicando e compondo funções.
É um paradigma declarativo, onde funções descrevem transformações de valores, em vez de uma sequência de instruções que atualizam o estado do programa.

Ao invés de variáveis mutáveis, temos valores imutáveis com funções que realizam mapeamento de um conjunto para outro.

<div style="display: flex; justify-content: center;">
    <img src="resources/map.png" width="250">
</div>

## Fundamentos de FP

### Funções como *first-class citizens*

Em Python, funções são *first-class citizens*, ou seja, são tratadas como qualquer outro valor.
Isso significa que podem ser:

- Atribuídas a variáveis

In [46]:
def add(x: int, y: int) -> int:
    return x + y

f = add
f(2, 3)

5

- Passadas como argumento

In [47]:
from typing import Callable


def apply(f: Callable[[int], int], x: int) -> int:
    return f(x)

apply(lambda x: x * 2, 5)

10

- Retornadas de outras funções

In [48]:
from typing import Callable


def make_adder(n: int) -> Callable[[int], int]:
    def add(x: int):
        return x + n
    return add

add5 = make_adder(5)
add5(3)

8

### Funções puras e impuras

Uma função pura sempre retorna o mesmo resultado para a mesma entrada e não causa efeitos colaterais.
Com isso, são fáceis de testar e de se raciocinar sobre.

In [44]:
# Pure function
def add(a: int, b: int) -> int:  
	return a + b

print(add(2, 2))
print(add(2, 2))

4
4


Funções impuras dependem ou modificam estado externo.
Com isso, podem ter resultados diferentes para o mesmo input.

In [45]:
total: int = 0  
  
def add_to_total(x: int) -> int:  
	global total  
	total += x
	return total

print(add_to_total(2))
print(add_to_total(2))

2
4


### Imutabilidade

Em FP, trabalhamos com valores não modificáveis e não com variáveis.

In [ ]:
from typing import List, TypeVar

T = TypeVar("T")

def add_item(lst: List[T], item: T) -> List[T]:  
	return lst + [item] # returns a new list

items = [1, 2, 3]

new_items = add_item(items, 4)

items.append(5)
print(new_items) # [1, 2, 3, 4]

[1, 2, 3, 4]


Isso difere de uma abordagem mutável, como a do exemplo a seguir:

In [ ]:
from typing import List, TypeVar

T = TypeVar("T")

def add_item(lst: List[T], item: T) -> List[T]:  
	lst.append(item)  
	return lst

items = [1, 2, 3]

new_items = add_item(items, 4) # both items and new_item refer to the same list

items.append(5)
print(new_items) # [1, 2, 3, 4, 5]

[1, 2, 3, 4, 5]


### Funções de ordem superior

Funções de ordem superior são funções que recebem ou retornam outras funções.

In [ ]:
from typing import Callable, TypeVar

T = TypeVar("T")
U = TypeVar("U")

def apply(f: Callable[[T], U], x: T) -> U:  
	return f(x)  
	  
apply(lambda x: x * 2, 5)

10

Algumas das funções de ordem superior mais comuns já estão nativamente disponíveis, alguns exemplos são:

In [12]:
list(map(lambda x: x * 2, [1, 2, 3]))

[2, 4, 6]

In [13]:
list(filter(lambda x: x % 2 == 1, [1, 2, 3, 4, 5]))

[1, 3, 5]

In [ ]:
from functools import reduce


reduce(lambda x, y: x + y, [1, 2, 3, 4, 5]) # ((((1 + 2) + 3) + 4) + 5).

15

### Composição de funções

Composição é combinar funções simples para formar funções mais complexas.

In [ ]:
def double(x: int) -> int:  
	return x * 2  
  
def increment(x: int) -> int:  
	return x + 1  
  
def compose(f: Callable[[int], int], g: Callable[[int], int]) -> Callable[[int], int]:  
	return lambda x: f(g(x))  
  
print(f"double(increment(3)) = {compose(double, increment)(3)}") # 8
print(f"increment(double(3)) = {compose(increment, double)(3)}") # 7

double(increment(3)) = 8
increment(double(3)) = 7


### Aplicação parcial

É quando passamos para uma função parte de seus argumentos e geramos uma nova função que precisa apenas dos argumentos restantes para ser executada.

Apesar de não ter auto [currying](https://fsharpforfunandprofit.com/posts/currying/), ainda podemos usar aplicação parcial.

In [ ]:
from functools import partial  
  
def multiply(a: int, b: int) -> int:  
    return a * b  
  
double = partial(multiply, 2)

print(double(3)) # 6
print(double(5)) # 10

6
10


In [ ]:
from toolz import curry   # type: ignore
  
@curry  
def add(a: int, b: int) -> int:  
	return a + b  
  
add5 = add(5)
add5(3) # type: ignore

8

## List Comprehensions e Generator Expressions

List comprehensions são uma ideia funcional, mas com sintaxe adaptada ao estilo Python.
É a aplicação de uma transformação combinada com um filtro em uma coleção.

A inspiração vem de Haskell:
``` Haskell
[x * 2 | x <- [1,2,3], x > 0]
```

In [40]:
result = [x * 2 for x in [1, 2, 3] if x > 0]

print(f"type={type(result)} | {result=}")

type=<class 'list'> | result=[2, 4, 6]


No entanto a operação ainda difere quanto a evaluação ser lazy em Haskell e eager em python, mas isso é intencional e python tem também uma implementação lazy, chamada de **generator expressions**:

In [ ]:
result = (x * 2 for x in [1, 2, 3] if x > 0) # lazy evaluated, each value is yield

print(f"type={type(result)} | {result=}")
print(next(result))
print(list(result))

type=<class 'generator'> | result=<generator object <genexpr> at 0x73eb3fdce400>
2
[4, 6]


## Pipelines

Pipelines são uma maneira de se aplicar uma sequência de transformações em um conjunto de dados.

Ao invés de chamar funções aninhadas:

`f(g(h(x)))`

Escrevemos transformações como um fluxo linear e legível:

`x → h → g → f`


No estilo imperativo, transformações são aplicadas sequencialmente, detalhando como isso deve ser feito:

In [49]:
from typing import List

data = [1, -2, 3, 0, 4]

filtered: List[int] = []
for x in data:
    if x > 0:
        filtered.append(x)

doubled: List[int] = []
for x in filtered:
    doubled.append(x * 2)

doubled

[2, 6, 8]

O que é verboso e apresenta risco de mutações de estado.

Normalmente em python usamos list comprehension, devido a esse ser um caso simples de filter -> map:

In [50]:
data = [1, -2, 3, 0, 4]

result = [x * 2 for x in data if x > 0]
result

[2, 6, 8]

E se o conjunto de dados for grande o bastante para precisarmos de maior eficiência, podemos utilizar generator expressions:

In [51]:
data = [1, -2, 3, 0, 4]

pipeline = (x for x in data if x > 0)
pipeline = (x * 2 for x in pipeline)

list(pipeline)

[2, 6, 8]

Podemos também usar libs como `toolz`:

In [52]:
from toolz import pipe # type: ignore

data = [1, -2, 3, 0, 4]

result = pipe( # type: ignore
    data,
    lambda xs: filter(lambda x: x > 0, xs), # type: ignore
    lambda xs: map(lambda x: x * 2, xs), # type: ignore
    list
)

result

[2, 6, 8]

Outra alternativa é a lib `returns`:

In [53]:
from returns.pipeline import flow

data = [1, -2, 3, 0, 4]

result = flow( # type: ignore
    data,
    lambda xs: [x for x in xs if x > 0], # type: ignore
    lambda xs: [x * 2 for x in xs], # type: ignore
)

result

[2, 6, 8]

O que também pode ser aproveitado em libs famosas como `pandas`:

In [54]:
import pandas as pd

df = pd.DataFrame({
    "value": [1, -2, 3, 0, 4]
})

(df
 .pipe(lambda d: d[d["value"] > 0])
 .pipe(lambda d: d.assign(value=d["value"] * 2))
)

,value
0,2
2,6
4,8


## Limitações de FP em python

### Contexto Histórico

Guido, o criador de python, resume bem como python se relaciona com FP em um post de 2009: [Origins of Python's "Functional" Features](https://python-history.blogspot.com/2009/04/origins-of-pythons-functional-features.html)

> I didn't view Python as a functional programming language. However, earlier on, it was clear that users wanted to do much more with lists and functions.

> Lambda was really only intended to be a syntactic tool for defining anonymous functions. However, the choice of terminology had many unintended consequences. For instance, users familiar with functional languages expected the semantics of lambda to match that of other languages. As a result, they found Python’s implementation to be sorely lacking in advanced features.

> Curiously, the map, filter, and reduce functions that originally motivated the introduction of lambda and other functional features have to a large extent been superseded by list comprehensions and generator expressions. In fact, the reduce function was removed from list of builtin functions in Python 3.0.

> Lastly, even though a number of functional programming features have been introduced over the years, Python still lacks certain features found in “real” functional programming languages.

Em outras palavras, python tem recursos de programação funcional, mas não é uma linguagem funcional e isso é intencional.

### Tail Call Optimization

Python não faz otimização de chamadas recursivas (tail call optimization).
O criador da linguagem cita como uma das justificativas manter o stack trace:
[Tail Recursion Elimination - 2009](https://neopythonic.blogspot.com/2009/04/tail-recursion-elimination.html)

### Pattern matching (não exaustivo)

Python tem `match`, mas não tem verificação de exaustividade.
Algo que pode ser feito para tentar se aproximar desse comportamento é alterar a configuração `typeCheckingMode` do pyright para o modo `strict`, com isso passamos a ver um warning em matchs incompletos.

In [26]:
from typing import Literal

Status = Literal["success", "error", "canceled"]

# if typeCheckingMode is set to strict in your linter you should see the error:
# Cases within match statement do not exhaustively handle all values
# Unhandled type: "Literal['canceled']"
def handle(status: Status):
    match status:
        case "success":
            return 1
        case "error":
            return 0

print(handle("success"))
print(handle("canceled"))

1
None


### Mutabilidade por padrão

Apesar de alguns tipos serem imutáveis por design:
- int
- float
- str
- tuple

Em Python, imutabilidade é uma convenção, não uma garantia.

In [28]:
n = 2
m = n
m += 2

print(f"{m=} e {n=}") # tipos primitivos são imutaveis

m=4 e n=2


É dito que é uma convenção porque mesmo que existam maneiras de se criar "tipos" imutaveis como:

In [35]:
from dataclasses import dataclass

@dataclass(frozen=True)
class User:
    name: str
    age: int

u = User("Olivia", 30)
print(u)
# u.age = 31 # resulta em um erro em runtime

User(name='Olivia', age=30)


A linguagem ainda permite burlar isso:

In [36]:
object.__setattr__(u, 'age', 31)
print(u)

User(name='Olivia', age=31)


### Lambdas limitadas

Lambdas em Python são restritas a uma única expressão.

O que nos impede de fazer algo como:

``` python
lambda x:
    if x < 0:
        return 0
    y = x * 2
    return y + 1
```

No lugar disso, devemos definir uma função:

In [ ]:
def process(x: int):
    if x < 0:
        return 0
    y = x * 2
    return y + 1

Com a adição de operadores ternarios passamos a poder fazer algo assim, mas não é recomendado por não ser "pythonico":

``` python
lambda x: 0 if x < 0 else (x * 2) + 1
```

### Limitações de sintaxe

Python não possui operadores nativos para composição ou pipeline.
Portanto não podemos escrever algumas operações comuns de modo simplificado:
``` python
h = f >> g
```
ou
``` python
h = 
    dados 
    |> filtrar 
    |> transformar 
    |> validar
```

## Consequências das limitações

FP ainda é relevante para python e é comum ver seu uso localizado, na forma de: 
- list comprehensions
- generator expressions
- funções tratadas como cidadão de primeira classe

Mas implementações globais são raras, devido a falta de adesão completa ao paradigma funcional por parte da linguagem.

## Conclusão e Recomendações

Foque no uso de funções puras, construindo funções menores e objetivas. Com isso não será necessário replicar o estado do sistema para executar testes, basta pensar em termos de entradas e saídas.

Use `mypy` para adicionar verificações de tipos ao seu código e priorize a redução de mutabilidade.

Escreva funções que possam ser compostas com facilidade, seja por meio de decoradores, pipelines ou encadeamento.

Prefira classes imutáveis, utilizando `dataclasses` ou `BaseModel` (Pydantic).

Use FP para implementar as regras de negócio e modelar o domínio de suas aplicações. O livro [Domain Modeling Made Functional](https://www.amazon.com/Domain-Modeling-Made-Functional-Domain-Driven/dp/1680502549) é uma ótima referência para isso.

Explore também os módulos nativos voltados a FP, listados na [documentação oficial](https://docs.python.org/3/library/functional.html).

Há ainda algumas bibliotecas úteis feitas pela comunidade:
- [Expression](https://github.com/dbrattli/Expression)
- [Toolz](https://github.com/pytoolz/toolz)
- [Returns](https://github.com/dry-python/returns)

Leituras recomendadas:
- [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html)
- *Functional Programming In Python* - David Mertz
- [Domain Modeling Made Functional - Scott Wlaschin](https://www.amazon.com/Domain-Modeling-Made-Functional-Domain-Driven/dp/1680502549)